In [ ]:
# import shutil
# from google.colab import drive
# drive.mount('/content/drive')

# shutil.unpack_archive('/content/drive/MyDrive/cnn_dataset.zip', '/content')
# print("cnn_dataset 압축 해제 완료")


In [ ]:
# =====================================================================
# 3단계: CNN 분류기 학습 (MobileNetV2 전이학습, freeze / 파인튜닝 두 버전)
# =====================================================================
# 2_crop_bboxes_for_cnn.py 로 만든 cnn_dataset/ 폴더를 사용합니다.
# cnn_dataset/train/plastic, cnn_dataset/train/glass, cnn_dataset/train/can ...
#
# freeze 버전과 파인튜닝 버전을 각각 학습해서 검증 정확도를 비교한 뒤,
# 더 나은 쪽을 최종 모델로 선택하면 됩니다. (Google Colab GPU 권장)
# =====================================================================

import os
os.environ["TF_USE_LEGACY_KERAS"] = "1"  # TF 2.16+ 기본 Keras 3와 TFLiteConverter 호환성 버그 회피

import tensorflow as tf
tf.config.set_visible_devices([], "GPU")  # RandomBrightness가 tensorflow-metal(GPU)에서 커널 크래시를 일으켜 CPU로 강제
from tensorflow.keras import layers, models

IMG_SIZE = (224, 224)
BATCH_SIZE = 32
EPOCHS = 15
DATA_DIR = "cnn_dataset"   # 2번 스크립트의 OUTPUT_DIR과 동일 경로로 수정

# --- 데이터 불러오기 ---
train_ds = tf.keras.utils.image_dataset_from_directory(
    f"{DATA_DIR}/train", image_size=IMG_SIZE, batch_size=BATCH_SIZE
)
val_ds = tf.keras.utils.image_dataset_from_directory(
    f"{DATA_DIR}/valid", image_size=IMG_SIZE, batch_size=BATCH_SIZE
)

class_names = train_ds.class_names
print("클래스:", class_names)  # 예: ['can', 'glass', 'plastic']

AUTOTUNE = tf.data.AUTOTUNE
train_ds = train_ds.cache().prefetch(buffer_size=AUTOTUNE)
val_ds = val_ds.cache().prefetch(buffer_size=AUTOTUNE)

# 데이터 증강: 위치가 살짝씩 달라지는 실제 환경 대비
augment = tf.keras.Sequential([
    layers.RandomFlip("horizontal"),
    layers.RandomRotation(0.1),
    layers.RandomZoom(0.1),
    layers.RandomBrightness(0.15),  # 조명 변화 대비 (유리·캔 반사 이슈 보완)
])
normalization = layers.Rescaling(1.0 / 127.5, offset=-1)  # MobileNetV2 표준 전처리


def build_model(base_trainable=False, fine_tune_at=100):
    base = tf.keras.applications.MobileNetV2(
        input_shape=IMG_SIZE + (3,), include_top=False, weights="imagenet"
    )
    base.trainable = base_trainable
    if base_trainable:
        # fine_tune_at 이전 레이어는 계속 고정, 이후 레이어만 재학습
        for layer in base.layers[:fine_tune_at]:
            layer.trainable = False

    inputs = tf.keras.Input(shape=IMG_SIZE + (3,))
    x = augment(inputs)
    x = normalization(x)
    x = base(x, training=False)
    x = layers.GlobalAveragePooling2D()(x)
    x = layers.Dropout(0.2)(x)
    outputs = layers.Dense(len(class_names), activation="softmax")(x)

    model = models.Model(inputs, outputs)
    lr = 1e-5 if base_trainable else 1e-3  # 파인튜닝은 학습률을 훨씬 낮게
    model.compile(
        optimizer=tf.keras.optimizers.Adam(lr),
        loss="sparse_categorical_crossentropy",
        metrics=["accuracy"],
    )
    return model


# --- 1) freeze 버전: 특징 추출부 고정, 마지막 분류층만 학습 ---
print("\n=== freeze 버전 학습 ===")
freeze_model = build_model(base_trainable=False)
freeze_history = freeze_model.fit(
    train_ds, validation_data=val_ds, epochs=EPOCHS, verbose=2
)
freeze_model.save("model_freeze.keras")

# --- 2) 파인튜닝 버전: 상위 레이어까지 낮은 학습률로 재학습 ---
print("\n=== 파인튜닝 버전 학습 ===")
finetune_model = build_model(base_trainable=True, fine_tune_at=100)
finetune_history = finetune_model.fit(
    train_ds, validation_data=val_ds, epochs=EPOCHS, verbose=2
)
finetune_model.save("model_finetune.keras")

# --- 3) 두 모델 검증 정확도 비교 ---
freeze_loss, freeze_acc = freeze_model.evaluate(val_ds, verbose=0)
finetune_loss, finetune_acc = finetune_model.evaluate(val_ds, verbose=0)

best_model = finetune_model if finetune_acc >= freeze_acc else freeze_model
best_name = "finetune" if finetune_acc >= freeze_acc else "freeze"

print("\n" + "=" * 50)
print("최종 결과 요약")
print("=" * 50)
print(f"클래스: {class_names}")
print(f"freeze   검증 정확도: {freeze_acc * 100:6.2f}%   (loss {freeze_loss:.4f})")
print(f"finetune 검증 정확도: {finetune_acc * 100:6.2f}%   (loss {finetune_loss:.4f})")
print(f"-> 최종 선택 모델: {best_name} 버전")
print("=" * 50)

# --- 4) 최종 모델을 .tflite로 변환 (Jetson Nano 등 경량 기기 배포용) ---
converter = tf.lite.TFLiteConverter.from_keras_model(best_model)
converter.optimizations = [tf.lite.Optimize.DEFAULT]  # 크기/속도 최적화
tflite_model = converter.convert()

with open("classifier.tflite", "wb") as f:
    f.write(tflite_model)

# 클래스 순서도 같이 저장해둬야 나중에 추론 결과 인덱스를 해석할 수 있음
with open("class_names.txt", "w", encoding="utf-8") as f:
    f.write("\n".join(class_names))

print("\n완료: classifier.tflite, class_names.txt 저장됨")

# =====================================================================
# 다음 단계
# 1) best.pt (YOLO), classifier.tflite (CNN), class_names.txt 를
#    Jetson Nano로 옮김
# 2) 실제 카메라로 YOLO 검출 -> 바운딩박스 크롭 -> CNN 추론 파이프라인 테스트
# 3) 정지-촬영 방식이면: IR센서 감지 -> 벨트 정지 -> 촬영 -> 이 파이프라인
#    -> 서보 매핑 -> 벨트 재가동 순서로 통합
# =====================================================================


/Users/ovo/Documents/KCCI/AI_project/Jetson_Recycling-conveyor-belt/.venv/lib/python3.9/site-packages/urllib3/__init__.py:35: NotOpenSSLWarning: urllib3 v2 only supports OpenSSL 1.1.1+, currently the 'ssl' module is compiled with 'LibreSSL 2.8.3'. See: https://github.com/urllib3/urllib3/issues/3020
  warnings.warn(


Found 402 files belonging to 3 classes.
Found 116 files belonging to 3 classes.
클래스: ['metal', 'paper', 'plastic']

=== freeze 버전 학습 ===


Epoch 1/15
13/13 - 22s - loss: 1.1672 - accuracy: 0.4428 - val_loss: 0.7118 - val_accuracy: 0.7241 - 22s/epoch - 2s/step
Epoch 2/15
13/13 - 3s - loss: 0.7610 - accuracy: 0.6592 - val_loss: 0.4489 - val_accuracy: 0.8534 - 3s/epoch - 226ms/step
Epoch 3/15
13/13 - 3s - loss: 0.4988 - accuracy: 0.8010 - val_loss: 0.3419 - val_accuracy: 0.9052 - 3s/epoch - 213ms/step
Epoch 4/15
13/13 - 3s - loss: 0.4001 - accuracy: 0.8433 - val_loss: 0.3019 - val_accuracy: 0.9224 - 3s/epoch - 222ms/step
Epoch 5/15
13/13 - 3s - loss: 0.3188 - accuracy: 0.8955 - val_loss: 0.2780 - val_accuracy: 0.9138 - 3s/epoch - 218ms/step
Epoch 6/15
13/13 - 3s - loss: 0.3065 - accuracy: 0.9005 - val_loss: 0.2572 - val_accuracy: 0.9224 - 3s/epoch - 219ms/step
Epoch 7/15
13/13 - 3s - loss: 0.2352 - accuracy: 0.9229 - val_loss: 0.2400 - val_accuracy: 0.9310 - 3s/epoch - 218ms/step
Epoch 8/15
13/13 - 3s - loss: 0.2598 - accuracy: 0.9129 - val_loss: 0.2432 - val_accuracy: 0.9397 - 3s/epoch - 217ms/step
Epoch 9/15
13/13 - 3s - l

Epoch 1/15
13/13 - 13s - loss: 1.2739 - accuracy: 0.3731 - val_loss: 0.9046 - val_accuracy: 0.5431 - 13s/epoch - 1s/step
Epoch 2/15
13/13 - 4s - loss: 0.8487 - accuracy: 0.5945 - val_loss: 0.6549 - val_accuracy: 0.7586 - 4s/epoch - 314ms/step
Epoch 3/15
13/13 - 4s - loss: 0.6645 - accuracy: 0.7065 - val_loss: 0.5023 - val_accuracy: 0.8362 - 4s/epoch - 304ms/step
Epoch 4/15
13/13 - 4s - loss: 0.5412 - accuracy: 0.7935 - val_loss: 0.4022 - val_accuracy: 0.8707 - 4s/epoch - 313ms/step
Epoch 5/15
13/13 - 4s - loss: 0.4319 - accuracy: 0.8532 - val_loss: 0.3359 - val_accuracy: 0.8879 - 4s/epoch - 307ms/step
Epoch 6/15
13/13 - 4s - loss: 0.3795 - accuracy: 0.8781 - val_loss: 0.2897 - val_accuracy: 0.9052 - 4s/epoch - 303ms/step
Epoch 7/15
13/13 - 4s - loss: 0.3439 - accuracy: 0.8831 - val_loss: 0.2557 - val_accuracy: 0.9483 - 4s/epoch - 305ms/step
Epoch 8/15
13/13 - 4s - loss: 0.3250 - accuracy: 0.8756 - val_loss: 0.2343 - val_accuracy: 0.9397 - 4s/epoch - 310ms/step
Epoch 9/15
13/13 - 4s - l

INFO:tensorflow:Assets written to: /var/folders/jt/jnmwv9t15h525db1wzbrn8580000gn/T/tmpd4s9zw0z/assets



완료: classifier.tflite, class_names.txt 저장됨


W0000 00:00:1785120830.002807 9187010 tf_tfl_flatbuffer_helpers.cc:390] Ignored output_format.
W0000 00:00:1785120830.003162 9187010 tf_tfl_flatbuffer_helpers.cc:393] Ignored drop_control_dependency.
2026-07-27 11:53:50.003748: I tensorflow/cc/saved_model/reader.cc:83] Reading SavedModel from: /var/folders/jt/jnmwv9t15h525db1wzbrn8580000gn/T/tmpd4s9zw0z
2026-07-27 11:53:50.017176: I tensorflow/cc/saved_model/reader.cc:51] Reading meta graph with tags { serve }
2026-07-27 11:53:50.017189: I tensorflow/cc/saved_model/reader.cc:146] Reading SavedModel debug info (if present) from: /var/folders/jt/jnmwv9t15h525db1wzbrn8580000gn/T/tmpd4s9zw0z
2026-07-27 11:53:50.130211: I tensorflow/compiler/mlir/mlir_graph_optimization_pass.cc:388] MLIR V1 optimization pass is not enabled
2026-07-27 11:53:50.146298: I tensorflow/cc/saved_model/loader.cc:234] Restoring SavedModel bundle.
2026-07-27 11:53:50.525570: I tensorflow/cc/saved_model/loader.cc:218] Running initialization op on SavedModel bundle at 